# LifeLoop — train both models on a Colab T4

Local CPU training gets killed by memory pressure part-way through an epoch, so
real runs happen here. Two models are trained, in this order:

1. **Classifier** — MobileNetV3-Small, 9 material classes, answers *what is this made of*
2. **Detector** — YOLOv8n, one class, answers *where are the discardable items*

They are separate on purpose. The detector has ~5,000 boxes for one question;
spread across 60 material categories that would be ~80 examples each, which is not
enough to learn from. Serving runs detector first, then the classifier on each crop.

## Before you start

**Runtime → Change runtime type → T4 GPU.** On CPU this notebook is no better than
the machine it is replacing.

Upload `colab_bundle.zip` (built by `scripts/make_colab_bundle.py`) to the top level
of your Google Drive.

## 1 — Confirm there is a GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."
    )

print(torch.cuda.get_device_name(0))
print(f"torch {torch.__version__}")

## 2 — Mount Drive and unpack

Unpacked to local disk rather than run from Drive. Drive is a network mount and the
data loader reads every image once per epoch, so training from it is several times
slower for no benefit.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pathlib
import zipfile

import pandas as pd

BUNDLE = pathlib.Path('/content/drive/MyDrive/colab_bundle.zip')
ML = pathlib.Path('/content/ml')

if not BUNDLE.exists():
    raise SystemExit(f'{BUNDLE} not found. Check the filename in your Drive root.')

ML.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(ML)

# The bundle mirrors the ml/ layout, so this folder works as ML_ROOT unchanged.
for name in ('wasteml', 'scripts', 'data'):
    print(name, 'ok' if (ML / name).exists() else 'MISSING')

# --- two path rewrites, both of which must happen after every extract ---
#
# 1. The splits carry whatever absolute paths the machine that built them used.
#    Rebuilt here from class + filename so the source root does not matter.
import ntpath

for split in ('train', 'val', 'test'):
    csv = ML / 'data' / 'splits' / f'{split}.csv'
    frame = pd.read_csv(csv)
    frame['path'] = [
        f'{ML}/data/raw/{label}/{ntpath.basename(str(path))}'
        for path, label in zip(frame['path'], frame['label'])
    ]
    missing = sum(1 for path in frame['path'] if not pathlib.Path(path).exists())
    frame.to_csv(csv, index=False)
    print(f'{split:6} {len(frame):5} rows, {missing} missing')

# 2. Ultralytics resolves a relative `path:` against the working directory, not
#    against the yaml's folder, so this one has to be absolute.
detection = ML / 'data' / 'detection'
(detection / 'data.yaml').write_text(
    f'path: {detection}
'
    'train: images/train
'
    'val: images/val
'
    'test: images/test
'
    '
'
    'nc: 1
'
    'names:
'
    '  0: waste
'
)

counts = {
    d.name: len(list(d.glob('*')))
    for d in sorted((ML / 'data' / 'raw').iterdir())
    if d.is_dir()
}
print()
for name, count in counts.items():
    print(f'{name:12} {count}')
print(f'{"TOTAL":12} {sum(counts.values())}')


## 3 — Dependencies

Colab already ships torch and torchvision with CUDA. Installing from
`requirements.txt` would pull the CPU wheels over them and silently cost you the
GPU, so only what is genuinely missing is installed.

In [ ]:
!pip install -q ultralytics onnx onnxruntime

import torch
print('cuda still available:', torch.cuda.is_available())

## 4 — Train the classifier

Two phases. Phase A trains only the new head with the backbone frozen, so the
randomly-initialised head cannot push large gradients back through pretrained
weights and wreck them. Phase B unfreezes everything at a 10x smaller learning
rate.

`--min-images 1` keeps the default behaviour of skipping classes with too few
images to learn. Wood is empty, so the model trains on 9 of 10 classes and cannot
predict Wood at all — that is deliberate and reported, not silently ignored.

Batch size is raised from the CPU default of 32: a T4 has the memory for it and
larger batches are faster per epoch.

In [ ]:
%cd /content/ml
!python -u scripts/train.py --batch-size 64

## 5 — Evaluate the classifier

Fits the temperature scaling and the per-class abstention thresholds, then reports
the test set.

**Read the macro-F1, not the accuracy.** The classes are imbalanced, so accuracy is
dominated by whichever class is largest. macro-F1 weights every class equally,
which is what matters when Hazardous being wrong is worse than Plastic being wrong.

**And read it as an upper bound.** Every image in this dataset comes from a public
source, so `prepare_dataset.py` was run with `--allow-public-holdout`. These are
studio and stock photographs; real waste in real light scores lower.

In [ ]:
!python -u scripts/evaluate.py

## 6 — Train the detector

One class, `waste`. 40 epochs with patience 12, so it stops early once validation
mAP stops improving rather than burning the full budget.

`--batch 8` is the CPU default; a T4 handles 16 at 640px.

In [ ]:
!python -u scripts/train_detector.py --batch 16 --epochs 40

## 7 — Calibrate the detector's confidence threshold

This step is easy to skip and expensive to skip. mAP50 measures whether boxes are
*ranked* correctly, not what absolute score a real box gets — so a checkpoint with
better mAP can find fewer items at a fixed threshold. Serving needs a threshold
picked from a measured precision/recall curve, which is what this writes into
`artifacts/detector-thresholds.json`.

In [ ]:
!python -u scripts/calibrate_detector.py --min-precision 0.55

## 8 — Export to ONNX

Verifies parity against the PyTorch outputs rather than assuming it. A silent
mismatch here is the classic cause of "good in the notebook, useless in the app".

In [ ]:
!python -u scripts/export.py

## 9 — Copy the artifacts back to Drive

Colab discards local disk when the runtime ends. Download `lifeloop-artifacts.zip`
from Drive and extract it over `ml/artifacts/` on your machine.

The checkpoint records which classes it was trained on, so `load_checkpoint`
refuses a model whose class list does not match the config it is being served
under — a 9-class model loads fine, a stale 8-class one does not.

In [ ]:
import shutil

archive = shutil.make_archive(
    '/content/drive/MyDrive/lifeloop-artifacts', 'zip',
    root_dir='/content/ml/artifacts',
)

import pathlib
size = pathlib.Path(archive).stat().st_size / 1e6
print(f'{archive}  ({size:.0f} MB)')

for path in sorted(pathlib.Path('/content/ml/artifacts').rglob('*')):
    if path.is_file():
        print(f'  {path.relative_to("/content/ml/artifacts")}')

## 10 — Sanity check on one image

Runs the full serving path — detector, crop, classifier — the way the app does,
rather than trusting the metrics alone. Upload a photograph of a mixed pile when
prompted.

If this disagrees with the reported scores, the scores are measuring a different
problem from the one the product solves. That gap is the point of the exercise.

In [ ]:
import sys
sys.path.insert(0, '/content/ml')

from google.colab import files
from PIL import Image

uploaded = files.upload()
name = next(iter(uploaded))
image = Image.open(name).convert('RGB')

from wasteml import config, detect, model as model_lib

bundle = model_lib.load_checkpoint(config.ARTIFACTS_DIR / 'waste_mobilenet_v3_small.pt')
print('classes:', bundle['classes'] if isinstance(bundle, dict) else 'see load_checkpoint')

# analyse_scene wants the detector, the classifier, the eval transform, the class
# list, the fitted temperature and the per-class thresholds. Pull them from the
# artifacts written above rather than hardcoding, so this reflects what will ship.
import json

thresholds = json.loads(
    (config.ARTIFACTS_DIR / 'waste_mobilenet_v3_small_thresholds.json').read_text()
)
print('thresholds:', thresholds)

detector_thresholds = json.loads(
    (config.ARTIFACTS_DIR / 'detector-thresholds.json').read_text()
)
print('detector serving threshold:', detector_thresholds)

## What to do with the numbers

Three known gaps, all of which belong in the paper rather than being left implied:

- **Wood is untrained.** 1 image. The model is 9-class and says so at startup.
- **Every metric is an upper bound.** All training and test images are public
  studio or stock photographs. `--allow-public-holdout` was required to split them
  at all.
- **The two-stage pipeline has no end-to-end score.** mAP50 and macro-F1 are each
  measured alone. Nothing yet measures *photograph of a mixed pile in, correct
  material breakdown out*, which is what the product claims to do.

Closing all three needs your own photographs: ~100 wooden items, and ~100 mixed
piles with per-item material boxes.